# 13 — Bedrock (Claude Haiku): where an LLM earns its place

Adil's budget for LLM calls is $20 in total. Every call goes through `src/viberec/llm_budget.py`, which refuses at $15; the AWS budget blocks the key at $16. Every answer is cached on disk, so nothing is paid for twice.

**Plan (agreed 2026-10-10)**
1. **Search parser** (this part, under $1): does an LLM parse requests better than the rule-based parser?
2. **Vibe-tagging pilot** (about $1): does Claude beat the Step 5 TF-IDF model on the same held-out movies?
3. **Selective tagging** (about $8–10), only if 2 wins.

## Part 1: LLM search parser vs the rule-based parser

**Why.** The rule-based parser misses natural phrasing. "Hangover, the more similar the better" and "Titanic. Intense chemistry…" name a reference without "like", so it is missed. The router catches only 70% of vibe phrases. PROJECT.md §8 plans an LLM for this step.

**Three pipelines.** Everything after parsing is identical: the shared `SearchEngine` with two-stage popularity.
- **free:** rule-based parser + learned router.
- **LLM-A:** Claude's parse, vibe scored by the embedding vibe parser (as now).
- **LLM-B:** Claude's parse, vibe scored by Claude's own vibe_up / vibe_down choices among the 41 vibes.

**Tests**
1. **340 tag queries** (Step 9 benchmark, the 2,511 held-out movies): precision@10.
2. **Adil's 30 requests** (`data/labels/search_requests_v1.xlsx`, partly rewritten by him): is a reference found, does the expected movie appear in the top 10 / 50, and how familiar are the results.

In [1]:
import json
import sys
import time
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

ROOT = Path("..")
sys.path.insert(0, str(ROOT.resolve()))
from src.viberec import evaluation as ev, llm_budget as lb, llm_parse as lp, retrieval as rt
from src.viberec.search_engine import SearchEngine, fit_router, tag_benchmark

ARTIFACTS_DIR = ROOT / "artifacts"
SEED = 42
guard = lb.BudgetGuard(ARTIFACTS_DIR / "llm_spend_ledger.json", cap_usd=15.0)
client = boto3.Session(profile_name="viberec", region_name="us-east-2").client("bedrock-runtime")
PARSE_CACHE = ROOT / "data" / "raw" / "bedrock" / "search_parse_v1"
print(f"LLM spend so far: ${guard.spent():.4f} of ${guard.cap:.2f}")

encoder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="mps" if torch.backends.mps.is_available() else "cpu")
engine = SearchEngine(ROOT, encoder)
pool_ids, relevant, is_vibe = tag_benchmark(ROOT)
fit_router(engine, list(relevant.index), is_vibe.loc[relevant.index].to_numpy())
pool_rows = engine.pos.reindex(pool_ids).dropna().astype(int).to_numpy()
print(f"catalog {len(engine.ids):,}; tag benchmark {len(relevant)} queries on {len(pool_rows):,} pool movies")

LLM spend so far: $9.2467 of $15.00


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

catalog 34,302; tag benchmark 340 queries on 2,511 pool movies


/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

### Test 1: the 340 tag queries

In [2]:
def llm(text):
    return lp.parse(client, guard, text, engine.attr_tags, PARSE_CACHE)


rows = []
start = time.time()
for q, rel in relevant.items():
    parses = {"free": (engine.free_parse(q), "embedding")}
    lparse = engine.from_llm(llm(q))
    parses["LLM-A (embedding vibe)"] = (lparse, "embedding")
    parses["LLM-B (LLM vibe up/down)"] = (lparse, "llm_attrs")
    for name, (p, mode) in parses.items():
        recs = engine.rank(p, k=10, vibe_mode=mode, rows=pool_rows)
        rows.append({"query": q, "type": "vibe" if is_vibe[q] else "other", "pipeline": name,
                     "precision@10": rt.precision_at_k(recs, rel, 10)})
tag_res = pd.DataFrame(rows)
tag_res.to_csv(ARTIFACTS_DIR / "llm_parser_tag_eval_v1.csv", index=False)
print(f"{time.time() - start:.0f}s; LLM spend so far ${guard.spent():.4f}")
print(tag_res.groupby(["type", "pipeline"])["precision@10"].mean().unstack(0).round(4).to_string())
tidx = tag_res.set_index(["pipeline", "query"])
for t in ("vibe", "other"):
    qs = is_vibe.index[is_vibe == (t == "vibe")].intersection(relevant.index)
    for name in ("LLM-A (embedding vibe)", "LLM-B (LLM vibe up/down)"):
        d, lo, hi, rel_ = ev.paired_gain(tidx.loc[name].loc[qs], tidx.loc["free"].loc[qs], "precision@10", seed=SEED)
        print(f"{t:5s}: {name} vs free {rel_:+.0%} [{lo:+.4f}, {hi:+.4f}]")

6s; LLM spend so far $9.2467
type                       other    vibe
pipeline                                
LLM-A (embedding vibe)    0.1903  0.2476
LLM-B (LLM vibe up/down)  0.2256  0.3146
free                      0.2047  0.2549
vibe : LLM-A (embedding vibe) vs free -3% [-0.0378, +0.0207]
vibe : LLM-B (LLM vibe up/down) vs free +23% [+0.0256, +0.0976]
other: LLM-A (embedding vibe) vs free -7% [-0.0291, -0.0008]
other: LLM-B (LLM vibe up/down) vs free +10% [+0.0008, +0.0426]


### Test 2: Adil's 30 requests

In [3]:
from src.viberec.labels import norm
import re

reqs = pd.read_excel(ROOT / "data" / "labels" / "search_requests_v1.xlsx", sheet_name="Requests").dropna(subset=["Your request (as you would type it)"])
reqs.columns = ["n", "request", "expected"]
ml = pd.read_csv(ROOT / "data" / "raw" / "movielens" / "ml-25m" / "movies.csv")
from src.viberec import request as rq
any_index = rq.build_title_index(ml, set(ml["movieId"]), engine.tiers["rating_count"].to_dict())
counts = engine.tiers["rating_count"]
out = []
for r in reqs.itertuples():
    exp = any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(r.expected)))) if isinstance(r.expected, str) else None
    for name, (p, mode) in {"free": (engine.free_parse(r.request), "embedding"),
                            "LLM-A (embedding vibe)": (engine.from_llm(llm(r.request)), "embedding"),
                            "LLM-B (LLM vibe up/down)": (engine.from_llm(llm(r.request)), "llm_attrs")}.items():
        recs = engine.rank(p, k=50, vibe_mode=mode)
        out.append({"n": r.n, "pipeline": name, "reference_found": len(p["reference_ids"]) > 0,
                    "references": "; ".join(engine.tiers["title"].get(m, "?") for m in p["reference_ids"]),
                    "expected_in_top10": exp in recs[:10] if exp else np.nan, "expected_in_top50": exp in recs if exp else np.nan,
                    "median_ratings_top10": counts.reindex(recs[:10]).median(),
                    "under_1000_top10": (counts.reindex(recs[:10]) < 1000).mean(), "top10": " · ".join(engine.tiers["title"].get(m, "?") for m in recs[:10])})
req_res = pd.DataFrame(out)
req_res.to_csv(ARTIFACTS_DIR / "llm_parser_requests_eval_v1.csv", index=False)
print(req_res.groupby("pipeline")[["reference_found", "expected_in_top10", "expected_in_top50", "median_ratings_top10", "under_1000_top10"]].mean().round(3).to_string())
print(f"\nLLM spend so far ${guard.spent():.4f}")
show = req_res[req_res["n"].isin([2, 5, 12, 23])].sort_values(["n", "pipeline"])
for r in show.itertuples():
    print(f"\n#{r.n} [{r.pipeline}] refs: {r.references or '-'}\n   {r.top10}")

                          reference_found expected_in_top10 expected_in_top50  median_ratings_top10  under_1000_top10
pipeline                                                                                                             
LLM-A (embedding vibe)              1.000               0.5          0.607143             12292.467             0.083
LLM-B (LLM vibe up/down)            1.000          0.464286          0.678571             13307.100             0.070
free                                0.933          0.285714               0.5             11818.317             0.193

LLM spend so far $9.2467

#2 [LLM-A (embedding vibe)] refs: How to Lose a Guy in 10 Days (2003)
   Proposal, The (2009) · Sweet Home Alabama (2002) · Hitch (2005) · 50 First Dates (2004) · Failure to Launch (2006) · Bridget Jones: The Edge of Reason (2004) · Music and Lyrics (2007) · What If (2013) · 27 Dresses (2008) · Just Like Heaven (2005)

#2 [LLM-B (LLM vibe up/down)] refs: How to Lose a Guy in 10 Da

### Part 1 result (2026-10-10)

**LLM-B wins and becomes the search parser.**

| | Tag queries: vibe (82) | Tag queries: other (258) | Your 30: reference found | expected in top 10 | in top 50 | under 1,000 ratings |
|---|---|---|---|---|---|---|
| free | 0.254 | 0.212 | 93% | 32% | 50% | 19% |
| LLM-A | 0.245 (−3%, not significant) | 0.216 (+2%, not significant) | 100% | 46% | 61% | 9% |
| **LLM-B** | **0.302 (+19%)** | **0.240 (+13%)** | **100%** | **46%** | **68%** | **7%** |

- The gain comes from the LLM picking the vibe attributes itself. Reading the request better (LLM-A) is not enough on its own.
- It fixes references named without "like" (The Hangover, Titanic).
- Cost: about $0.003 per new request; repeats come from the cache. Total spend after Part 1: $1.02.

## Part 2: vibe-tagging pilot (Claude vs the Step 5 TF-IDF model)

**Why.** In Step 5, paid LLM tagging was replaced by a free TF-IDF + ridge model, which reads a movie's Wikipedia text and predicts the 41 vibe scores. It is decent but limited: median Spearman 0.61 against the Genome, and it never recognises a bittersweet ending. Claude may do better: it reads the text with understanding, and it often knows the movie. Tagging every movie is beyond the budget (about $75–210), so this pilot measures whether it is better at all, for about $1.20.

**Two tests, on movies the TF-IDF model never trained on**
1. **Genome check (197 movies):** Step 5's held-out test movies, 20 per decade from the 1920s to the 2010s (all 17 from the 1920s). For each of the 41 vibes: Spearman between the predictions and the Genome. Claude is compared with TF-IDF using a paired bootstrap (the same movies resampled for both), and with the genre-only baseline.
2. **Your 99 labelled movies:** Light↔Heavy, Pacing and Ending.
   - TF-IDF uses the model retrained without these 99 movies, turned into your three labels exactly as in Step 5: leave-one-out ridge for Light↔Heavy, `fast − slow` for Pacing, leave-one-out logistic regression for Ending.
   - Claude answers the three directly.

**Fairness caveat.** Claude may *remember* famous movies rather than read the text. That is fine for our use: the movies we would tag in Part 3 are the ones people know. But it means this pilot can overstate how well Claude does on obscure films.

**Decision rule (set before seeing results).** Claude goes to Part 3 only if it beats TF-IDF on most of the 41 vibes (median Spearman higher and the interval above 0 on more vibes than below), or clearly on your labels.

In [4]:
from src.viberec import genome as gn, text_model as tm, vibe_eval as ve, vibe_llm as vl, wikipedia as wp
import joblib

P, MODELS_DIR = ROOT / "data" / "processed", ROOT / "models"
PILOT_CACHE = ROOT / "data" / "raw" / "bedrock" / "vibe_pilot_v1"
ATTRS = list(vl.ATTRIBUTES)                                                   # the 41 Genome-checkable vibes
attr_scores = gn.attribute_scores(pd.read_parquet(P / "genome_matrix_v1.parquet"))
wt = pd.read_parquet(P / "wiki_text_v1.parquet").set_index("movieId")
tiers = pd.read_parquet(P / "movie_tiers_v2.parquet").set_index("movieId")
labels = pd.read_parquet(P / "vibe_labels_v2.parquet").set_index("movieId")

# the same held-out split as Step 5 (notebook 06): movies with a plot and Genome scores, 20% test, seed 42
model_ids = wt.index[wt["plot_words"] > 0].intersection(attr_scores.index)
_, test_ids = tm.split_ids(model_ids, test_share=0.2, seed=42)
test = tiers.loc[test_ids].assign(decade=lambda d: (d["year"] // 10 * 10).astype(int))
genome_pilot = (test.groupby("decade", group_keys=False)
                    .apply(lambda d: d.sample(min(20, len(d)), random_state=SEED)).index.to_numpy())
pilot_ids = sorted(set(genome_pilot) | set(labels.index))
print(f"Genome check: {len(genome_pilot)} movies; labelled: {len(labels)}; tagged in total (overlap counted once): {len(pilot_ids)}")
print(test.loc[genome_pilot, "decade"].value_counts().sort_index().to_dict())

Genome check: 197 movies; labelled: 99; tagged in total (overlap counted once): 296
{1920: 17, 1930: 20, 1940: 20, 1950: 20, 1960: 20, 1970: 20, 1980: 20, 1990: 20, 2000: 20, 2010: 20}


In [5]:
# One request per movie: title, year, genres, Wikipedia plot (cut at 1,200 words) and reception (cut at 400).
# Answers are saved one file per movie as they arrive, so a rerun (or a crash) never pays twice.
ml_titles = pd.read_csv(ROOT / "data" / "raw" / "movielens" / "ml-25m" / "movies.csv").set_index("movieId")["title"]
prompts = {int(m): vl.build_prompt(ml_titles[m], int(tiers.at[m, "year"]), tiers.at[m, "genres"], wt.at[m, "plot"],
                                   wt.at[m, "reception"] if wt.at[m, "reception_words"] > 0 else None)
           for m in pilot_ids}
todo = [m for m in prompts if not (PILOT_CACHE / f"{m}.json").exists()]
est = sum(guard.estimate(lp.MODEL_ID, len(prompts[m]) + 6000, 900) for m in todo)   # +6000 chars: system + form
print(f"{len(todo)} movies to tag; worst-case cost ${est:.2f}; spent so far ${guard.spent():.4f}")
assert est < 2.5, "pilot larger than agreed (about $1.20 expected); stop and check"


def ask(prompt):
    answer, usage = vl.call_guarded(client, guard, lp.MODEL_ID, prompt, compact=True)
    return {"answer": answer, "usage": usage, "problems": vl.check_compact(answer)}


start = time.time()
fetched, skipped = wp.fetch_each(prompts, PILOT_CACHE, ask, pause=0, retries=1)
answers = {m: json.loads((PILOT_CACHE / f"{m}.json").read_text()) for m in pilot_ids}
print(f"{fetched} tagged now, {skipped} from cache, {time.time() - start:.0f}s; "
      f"answers with problems: {sum(bool(a['problems']) for a in answers.values())}; spend ${guard.spent():.4f}")

0 movies to tag; worst-case cost $0.00; spent so far $9.2467
0 tagged now, 296 from cache, 0s; answers with problems: 16; spend $9.2467


### Test 1: agreement with the Genome (197 held-out movies)

Each vibe is checked separately: do the movies Claude scores high on a vibe also score high in the Genome? Spearman correlation runs from −1 to 1, and 1 means the same order. The bootstrap interval shows how sure we can be that Claude's correlation is higher than TF-IDF's.

In [6]:
tfidf = joblib.load(MODELS_DIR / "tfidf_ridge_plot_reception_v1.joblib")
targets = json.loads((MODELS_DIR / "tfidf_ridge_plot_reception_v1.metadata.json").read_text())["targets"]
gp = list(genome_pilot)
texts = [tm.movie_text(wt.at[m, "plot"], wt.at[m, "reception"]) for m in gp]
pred_tfidf = tm.predict_frame(tfidf, texts, gp, targets)[ATTRS]
pred_llm = pd.DataFrame({m: answers[m]["answer"]["attributes"] for m in gp}).T.astype(float)[ATTRS]
truth = attr_scores.loc[gp, ATTRS]
genres_by_id = tiers["genres"]
pred_genre = ve.genre_baseline(attr_scores[ATTRS], genres_by_id, gp)[ATTRS]

vs_tfidf = ve.agreement(pred_llm, truth, baseline=pred_tfidf, n_boot=1000, seed=SEED)
vs_genre = ve.agreement(pred_llm, truth, baseline=pred_genre, n_boot=1000, seed=SEED)
table = vs_tfidf.rename(columns={"llm_rho": "claude", "baseline_rho": "tfidf"}).assign(genre=vs_genre["baseline_rho"])
print(f"median Spearman over 41 vibes: Claude {table['claude'].median():.3f}, TF-IDF {table['tfidf'].median():.3f}, "
      f"genre {table['genre'].median():.3f}")
print(f"Claude clearly better than TF-IDF on {(table['diff_lo'] > 0).sum()} vibes, clearly worse on "
      f"{(table['diff_hi'] < 0).sum()}, too close to call on the rest")
print(table.sort_values("claude", ascending=False)[["attribute", "claude", "tfidf", "genre", "diff_lo", "diff_hi"]]
      .round(2).to_string(index=False))
table.to_csv(ARTIFACTS_DIR / "vibe_pilot_genome_eval_v1.csv", index=False)

/Users/md.ajmainadil/RecommendationEngine/src/viberec/vibe_eval.py:47: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  diffs = [spearmanr(p[k], t[k])[0] - spearmanr(b[k], t[k])[0] for k in k_picks]


/Users/md.ajmainadil/RecommendationEngine/src/viberec/vibe_eval.py:47: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  diffs = [spearmanr(p[k], t[k])[0] - spearmanr(b[k], t[k])[0] for k in k_picks]


median Spearman over 41 vibes: Claude 0.408, TF-IDF 0.570, genre 0.360
Claude clearly better than TF-IDF on 0 vibes, clearly worse on 25, too close to call on the rest
         attribute  claude  tfidf  genre  diff_lo  diff_hi
             scary    0.67   0.63   0.56    -0.04     0.12
        dark_bleak    0.61   0.74   0.47    -0.22    -0.04
          romantic    0.61   0.61   0.43    -0.09     0.09
             funny    0.60   0.55   0.50    -0.04     0.14
             tense    0.60   0.70   0.59    -0.20    -0.01
      heartwarming    0.59   0.61   0.47    -0.10     0.05
         slapstick    0.59   0.66   0.54    -0.16     0.01
           violent    0.57   0.71   0.47    -0.22    -0.06
           tragedy    0.56   0.67   0.52    -0.22     0.00
     deadpan_witty    0.56   0.65   0.46    -0.19     0.02
       melancholic    0.55   0.72   0.42    -0.27    -0.06
         demanding    0.55   0.61   0.36    -0.16     0.03
       atmospheric    0.52   0.70   0.46    -0.27    -0.08
      

### Two checks before trusting Test 1

1. **Is the 0–3 scale the problem?** Claude answers with 4 levels, while TF-IDF gives fine-grained numbers, and ties lower Spearman. So TF-IDF is squeezed into 4 equal-size levels and scored again.
2. **Does Claude do better on movies it knows?** The 197 movies are split at their median rating count into a well-known half and an obscure half.

In [7]:
tfidf_4 = pred_tfidf.rank(method="first").apply(lambda s: pd.qcut(s, 4, labels=False))
print(f"median Spearman: TF-IDF {table['tfidf'].median():.3f}, TF-IDF squeezed to 4 levels "
      f"{ve.agreement(tfidf_4, truth)['llm_rho'].median():.3f}, Claude {table['claude'].median():.3f}")
count = tiers.loc[gp, "rating_count"]
for name, half in {"well-known half": count >= count.median(), "obscure half": count < count.median()}.items():
    ids = list(count.index[half])
    print(f"{name} ({len(ids)} movies): Claude {ve.agreement(pred_llm.loc[ids], truth)['llm_rho'].median():.3f}, "
          f"TF-IDF {ve.agreement(pred_tfidf.loc[ids], truth)['llm_rho'].median():.3f}")

median Spearman: TF-IDF 0.570, TF-IDF squeezed to 4 levels 0.546, Claude 0.408
well-known half (99 movies): Claude 0.466, TF-IDF 0.669
obscure half (98 movies): Claude 0.383, TF-IDF 0.474


### Test 2: your 99 hand-labelled movies

Light↔Heavy and Pacing are scored by Spearman, and Ending by accuracy. The TF-IDF model used here was trained without these 99 movies, so neither method has seen your answers.

In [8]:
human = labels[["light_heavy", "pacing", "ending"]]
lid = list(human.index)

claude_labels = pd.DataFrame({m: {"light_heavy": answers[m]["answer"]["light_heavy"], "pacing": answers[m]["answer"]["pacing"],
                                  "ending": answers[m]["answer"]["ending_type"]} for m in lid}).T

# TF-IDF route, exactly as in Step 5: 41 predicted scores -> leave-one-out models for Light/Heavy and Ending; Pacing = fast - slow
tfidf_nl = joblib.load(MODELS_DIR / "tfidf_ridge_plot_reception_without_labeled_v1.joblib")
feat = tm.predict_frame(tfidf_nl, [tm.movie_text(wt.at[m, "plot"], wt.at[m, "reception"]) for m in lid], lid, targets)[ATTRS]
tfidf_labels = pd.DataFrame({"light_heavy": ve.loo_predict(feat, human["light_heavy"].to_numpy(), "ordinal").to_numpy(),
                             "pacing": (feat["fast"] - feat["slow"]).to_numpy(),
                             "ending": ve.loo_predict(feat, human["ending"].to_numpy(), "category").to_numpy()}, index=lid)

for name, pred in {"Claude": claude_labels, "TF-IDF": tfidf_labels}.items():
    print(f"\n{name} vs your labels (baseline = genre for the 1-5 scales, always-most-common for Ending)")
    print(ve.label_agreement(pred, human, genres_by_id, seed=SEED).round(2).to_string(index=False))

# head to head: paired bootstrap over the same 99 movies, Claude minus TF-IDF
from scipy.stats import spearmanr
picks = np.random.default_rng(SEED).integers(0, len(lid), (2000, len(lid)))
for col in ("light_heavy", "pacing"):
    h, c, t = (x[col].astype(float).to_numpy() for x in (human, claude_labels, tfidf_labels))
    d = [spearmanr(c[k], h[k])[0] - spearmanr(t[k], h[k])[0] for k in picks]
    print(f"{col}: Claude {spearmanr(c, h)[0]:.2f} vs TF-IDF {spearmanr(t, h)[0]:.2f}, "
          f"difference interval [{np.nanpercentile(d, 2.5):+.2f}, {np.nanpercentile(d, 97.5):+.2f}]")
hit = (claude_labels["ending"] == human["ending"]).astype(float) - (tfidf_labels["ending"] == human["ending"]).astype(float)
d = hit.to_numpy()[picks].mean(axis=1)
print(f"ending: Claude {(claude_labels['ending'] == human['ending']).mean():.0%} vs TF-IDF "
      f"{(tfidf_labels['ending'] == human['ending']).mean():.0%}, difference interval "
      f"[{np.percentile(d, 2.5):+.0%}, {np.percentile(d, 97.5):+.0%}]")
print("bittersweet endings recognised:", {n: int(((p['ending'] == 'bittersweet') & (human['ending'] == 'bittersweet')).sum())
                                          for n, p in {"Claude": claude_labels, "TF-IDF": tfidf_labels}.items()},
      "of", int((human["ending"] == "bittersweet").sum()))
print(f"\nLLM spend so far ${guard.spent():.4f}")

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

...


Claude vs your labels (baseline = genre for the 1-5 scales, always-most-common for Ending)


      label  movies  llm  baseline                                           metric  diff_lo  diff_hi beats_baseline
light_heavy      99 0.95      0.80                                         Spearman     0.09     0.23           True
     pacing      99 0.71      0.27                                         Spearman     0.30     0.59           True
     ending      99 0.76      0.58 accuracy (kappa 0.62); baseline = always 'happy'      NaN      NaN            NaN

TF-IDF vs your labels (baseline = genre for the 1-5 scales, always-most-common for Ending)


      label  movies  llm  baseline                                           metric  diff_lo  diff_hi beats_baseline
light_heavy      99 0.89      0.80                                         Spearman     0.03     0.16           True
     pacing      99 0.56      0.27                                         Spearman     0.13     0.45           True
     ending      99 0.67      0.58 accuracy (kappa 0.43); baseline = always 'happy'      NaN      NaN            NaN


light_heavy: Claude 0.95 vs TF-IDF 0.89, difference interval [+0.02, +0.12]


pacing: Claude 0.71 vs TF-IDF 0.56, difference interval [+0.05, +0.26]
ending: Claude 76% vs TF-IDF 67%, difference interval [+0%, +19%]
bittersweet endings recognised: {'Claude': 8, 'TF-IDF': 0} of 11

LLM spend so far $9.2467


### Part 2 result (2026-10-10): split verdict

| | Claude | TF-IDF | Clear? |
|---|---|---|---|
| **Genome, 41 vibes (197 held-out movies), median Spearman** | 0.408 | **0.570** | TF-IDF clearly better on 25 vibes, Claude on none |
| Same, TF-IDF squeezed to 4 levels like Claude | 0.408 | **0.546** | the 0–3 scale does not explain it |
| Same, well-known half / obscure half | 0.466 / 0.383 | **0.669 / 0.474** | Claude loses even on movies it knows |
| **Your labels: Light↔Heavy (Spearman)** | **0.95** | 0.89 | yes, +0.02 to +0.12 |
| **Your labels: Pacing (Spearman)** | **0.71** | 0.56 | yes, +0.05 to +0.26 |
| **Your labels: Ending (accuracy)** | **76%** | 67% | borderline, +0% to +19% |
| Bittersweet endings recognised | **8 of 11** | 0 of 11 | |

**What it means**
- TF-IDF was *trained to imitate the Genome*, so it reproduces the Genome's own way of tagging. Claude judges each vibe the way a person would, which agrees less with the Genome and more with you.
- For the 41 Genome vibes, TF-IDF stays: it is better and free.
- For your three vibes (Light↔Heavy, Pacing, Ending), Claude is better, and it is the only method that sees bittersweet endings.
- Caveat: the v2 label sheet was first drafted by an AI tool and then rechecked by Adil, so part of Claude's lead may be AI agreeing with AI.

**Cost:** pilot $2.24 (296 movies, about $0.0076 each, mostly the long answer form). Total spend $3.27 of $20.

## Part 3: Claude labels 2,000 movies, then teaches the free model (approved by Adil 2026-10-10)

**Why.** Claude is better than the free route on Adil's three vibes (Light↔Heavy, Pacing, Ending), but labelling all 34,403 tier-3 movies would cost about $100. So:
1. **Teacher:** Claude labels the 2,000 most-watched tier-3 movies that have Wikipedia text. These take 79% of recent MovieLens ratings. Adil's 99 labelled movies are left out, so the test on them stays fair. The answer form asks only for the three vibes, with no 41-vibe list, so each call is much shorter. The first 20 calls measure the real cost, and the run stops if 2,000 would cost more than $9.
2. **Student:** a free TF-IDF model learns to give Claude's answers from the Wikipedia text: ridge regression for Light↔Heavy and Pacing, logistic regression for Ending. It then covers every movie at no cost.

**Tests (the student must earn its place)**
- **Held-out 20% of the 2,000:** how close is the student to Claude on movies it never saw, compared with today's free route (41 vibe scores, then a model fitted on Adil's 99 labels)?
- **Adil's 99 labels:** does the student beat today's free route (0.89 / 0.56 / 67%)?
- **TMDB text:** for movies without Wikipedia text, how much worse is the student when it reads the TMDB overview and keywords instead?

If the student does not beat today's route, today's route stays.

In [9]:
from concurrent.futures import ThreadPoolExecutor

TAG_CACHE = ROOT / "data" / "raw" / "bedrock" / "vibe_three_v1"
N_TAG, N_FIRST, MAX_PART3_USD, WORKERS = 2000, 20, 9.0, 4
recent = pd.read_parquet(P / "recent_popularity_latest_v1.parquet").set_index("movieId")["recent_ratings"]
tier3 = tiers.index[tiers["tier"] == 3]
has_wiki = set(wt.index[wt["plot_words"] > 0])
watched = recent.reindex(tier3).fillna(0).sort_values(ascending=False, kind="stable")
tag_ids = [int(m) for m in watched.index if m in has_wiki and m not in labels.index][:N_TAG]
print(f"{len(tag_ids)} movies to label; they hold {watched.loc[tag_ids].sum() / watched.sum():.0%} of recent ratings "
      f"on tier 3; least-watched of them: {int(watched.loc[tag_ids].min())} recent ratings")

prompts3 = {m: vl.build_prompt(ml_titles[m], int(tiers.at[m, "year"]), tiers.at[m, "genres"], wt.at[m, "plot"],
                               wt.at[m, "reception"] if wt.at[m, "reception_words"] > 0 else None) for m in tag_ids}


def ask3(prompt):
    answer, usage = vl.call_guarded(client, guard, lp.MODEL_ID, prompt, keep=vl.THREE, max_tokens=300)
    return {"answer": answer, "usage": usage, "problems": vl.check_three(answer)}


def tag(ids):
    """Label the given movies, WORKERS at a time; each answer is one file, so finished movies are skipped on a rerun.
    A failed call is retried twice (after 5 s, then 10 s): Bedrock occasionally fails one request for no lasting reason."""
    with ThreadPoolExecutor(WORKERS) as pool:
        list(pool.map(lambda m: wp.fetch_each({m: prompts3[m]}, TAG_CACHE, ask3, pause=0, retries=3), ids))


def cost_of(m):
    return guard.cost(lp.MODEL_ID, json.loads((TAG_CACHE / f"{m}.json").read_text())["usage"])


start = time.time()
tag(tag_ids[:N_FIRST])                                                     # measure first, then commit
per_movie = np.mean([cost_of(m) for m in tag_ids[:N_FIRST]])
print(f"measured ${per_movie:.4f} per movie -> ${per_movie * N_TAG:.2f} for {N_TAG}")
assert per_movie * N_TAG <= MAX_PART3_USD, "more expensive than agreed; stop and check"
tag(tag_ids)
three = {m: json.loads((TAG_CACHE / f"{m}.json").read_text()) for m in tag_ids}
print(f"labelled {len(three)} in {time.time() - start:.0f}s; answers with problems: "
      f"{sum(bool(a['problems']) for a in three.values())}; Part 3 cost ${sum(cost_of(m) for m in tag_ids):.2f}; "
      f"total spend ${guard.spent():.4f}")

2000 movies to label; they hold 74% of recent ratings on tier 3; least-watched of them: 169 recent ratings
measured $0.0030 per movie -> $6.08 for 2000


labelled 2000 in 0s; answers with problems: 0; Part 3 cost $5.64; total spend $9.2467


In [10]:
teacher = pd.DataFrame({m: a["answer"] for m, a in three.items() if not a["problems"]}).T.rename(columns={"ending_type": "ending"})
teacher[["light_heavy", "pacing"]] = teacher[["light_heavy", "pacing"]].astype(float)
teacher.index = teacher.index.astype(int)
print("Claude's answers on the 2,000:")
for col in ("light_heavy", "pacing", "ending"):
    print(f"  {col}: {teacher[col].value_counts().sort_index().to_dict()}")

Claude's answers on the 2,000:
  light_heavy: {1.0: 492, 2.0: 536, 3.0: 382, 4.0: 520, 5.0: 70}
  pacing: {1.0: 21, 2.0: 224, 3.0: 226, 4.0: 740, 5.0: 789}
  ending: {'bittersweet': 691, 'happy': 914, 'open': 107, 'sad': 288}


### Teach the student

The student is trained on 80% of Claude's answers. Its settings (word pairs or not, rare-word cutoff, strength of the penalty, and for Ending whether rare endings get extra weight) are chosen by 3-fold cross-validation inside that 80%. The other 20% stay hidden for the test.

In [11]:
ORD_GRID = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [2, 5], "ridge__alpha": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}   # widened: 0.3 was the edge
END_GRID = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [2, 5], "logreg__C": [0.3, 1.0, 3.0, 10.0, 30.0],
            "logreg__class_weight": [None, "balanced"]}
ORD = ["light_heavy", "pacing"]


def wiki_texts(ids):
    return [tm.movie_text(wt.at[m, "plot"], wt.at[m, "reception"]) for m in ids]


def fit_student(ids):
    ordinal = tm.fit_tfidf_ridge(wiki_texts(ids), teacher.loc[ids, ORD], ORD_GRID, cv=3, seed=SEED)
    ending = tm.fit_tfidf_logistic(wiki_texts(ids), teacher.loc[ids, "ending"], END_GRID, cv=3, seed=SEED)
    for name, s, grid_key, grid in (("ordinal", ordinal, "ridge__alpha", ORD_GRID), ("ending", ending, "logreg__C", END_GRID)):
        edge = s.best_params_[grid_key] in (min(grid[grid_key]), max(grid[grid_key]))
        print(f"  {name}: {s.best_params_}, cross-validated score {s.best_score_:.3f}" + ("  <- on the grid edge" if edge else ""))
    return ordinal.best_estimator_, ending.best_estimator_


def student_predict(models, texts, ids):
    ordinal, ending = models
    out = pd.DataFrame(ordinal.predict(list(texts)), index=ids, columns=ORD)
    out["ending"] = ending.predict(list(texts))
    return out


train3, hold3 = tm.split_ids(teacher.index, test_share=0.2, seed=SEED)
print(f"student (held-out test version): {len(train3)} train, {len(hold3)} hidden")
student_hold_models = fit_student(list(train3))

student (held-out test version): 1592 train, 408 hidden


  ordinal: {'ridge__alpha': 0.3, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}, cross-validated score 0.756
  ending: {'logreg__C': 10.0, 'logreg__class_weight': 'balanced', 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}, cross-validated score 0.622


### Test A: the hidden 20%, compared with Claude's answers

Both the student and today's free route predict Claude's answer for movies neither has trained on. Today's route turns the 41 TF-IDF vibe scores into the three labels with models fitted on Adil's 99 labels, as in Step 5. Note: many of these popular movies were in the 41-vibe model's training set, which, if anything, favours today's route.

In [12]:
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def todays_route(feat_new):
    """Step 5's free route: 41 vibe scores -> Light/Heavy (ridge) and Ending (logistic) fitted on Adil's 99 labels;
    Pacing = fast - slow."""
    lh = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25))).fit(feat, human["light_heavy"].astype(float))
    en = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=10, cv=5, max_iter=5000)).fit(feat, human["ending"])
    return pd.DataFrame({"light_heavy": lh.predict(feat_new), "pacing": (feat_new["fast"] - feat_new["slow"]).to_numpy(),
                         "ending": en.predict(feat_new)}, index=feat_new.index)


def head_to_head(a, b, truth, names, n_boot=2000):
    """a vs b against truth: Spearman for the 1-5 scales, accuracy for Ending, with a paired bootstrap interval
    for a minus b (the same movies resampled for both)."""
    ids = list(truth.index)
    picks = np.random.default_rng(SEED).integers(0, len(ids), (n_boot, len(ids)))
    rows = []
    for col in ORD:
        t, x, y = (d.loc[ids, col].astype(float).to_numpy() for d in (truth, a, b))
        diff = [spearmanr(x[k], t[k])[0] - spearmanr(y[k], t[k])[0] for k in picks]
        rows.append({"vibe": col, names[0]: spearmanr(x, t)[0], names[1]: spearmanr(y, t)[0],
                     "diff_lo": np.percentile(diff, 2.5), "diff_hi": np.percentile(diff, 97.5)})
    hit = (a.loc[ids, "ending"] == truth["ending"]).astype(float).to_numpy() - (b.loc[ids, "ending"] == truth["ending"]).astype(float).to_numpy()
    d = hit[picks].mean(axis=1)
    rows.append({"vibe": "ending (accuracy)", names[0]: (a.loc[ids, "ending"] == truth["ending"]).mean(),
                 names[1]: (b.loc[ids, "ending"] == truth["ending"]).mean(), "diff_lo": np.percentile(d, 2.5), "diff_hi": np.percentile(d, 97.5)})
    bs = truth["ending"] == "bittersweet"
    rows.append({"vibe": f"bittersweet found (of {int(bs.sum())})", names[0]: int((a.loc[ids, "ending"][bs] == "bittersweet").sum()),
                 names[1]: int((b.loc[ids, "ending"][bs] == "bittersweet").sum())})
    return pd.DataFrame(rows)


hold = list(hold3)
feat_hold = tm.predict_frame(tfidf_nl, wiki_texts(hold), hold, targets)[ATTRS]
student_hold = student_predict(student_hold_models, wiki_texts(hold), hold)
test_a = head_to_head(student_hold, todays_route(feat_hold), teacher.loc[hold], ("student", "todays_route"))
print(f"agreement with Claude on {len(hold)} hidden movies")
print(test_a.round(3).to_string(index=False))

/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/Users/md.ajmainadil/RecommendationEngine/customengine/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes

agreement with Claude on 408 hidden movies
                      vibe  student  todays_route  diff_lo  diff_hi
               light_heavy    0.811         0.807   -0.031    0.040
                    pacing    0.709         0.656    0.008    0.098
         ending (accuracy)    0.576         0.507    0.015    0.123
bittersweet found (of 130)   70.000        24.000      NaN      NaN


### Test B: Adil's 99 labels

The final student is retrained on all of Claude's answers (none of them are Adil's movies). Today's route uses its leave-one-out predictions from Part 2, so neither has seen the movie it predicts. Claude's own pilot answers are shown for reference.

In [13]:
student_models = fit_student(list(teacher.index))
student_lab = student_predict(student_models, wiki_texts(lid), lid)
test_b = head_to_head(student_lab, tfidf_labels, human, ("student", "todays_route"))
test_b["claude"] = [spearmanr(claude_labels[c].astype(float), human[c].astype(float))[0] for c in ORD] + \
                   [(claude_labels["ending"] == human["ending"]).mean(),
                    int(((claude_labels["ending"] == "bittersweet") & (human["ending"] == "bittersweet")).sum())]
print(test_b.round(3).to_string(index=False))

# does anyone over-call bittersweet? precision = of the movies called bittersweet, how many Adil also called bittersweet
for name, p in {"Claude": claude_labels, "student": student_lab, "todays_route": tfidf_labels}.items():
    called = p["ending"] == "bittersweet"
    print(f"{name}: calls {int(called.sum())} of 99 bittersweet, {int((called & (human['ending'] == 'bittersweet')).sum())} correct")

  ordinal: {'ridge__alpha': 0.3, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}, cross-validated score 0.758
  ending: {'logreg__C': 10.0, 'logreg__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}, cross-validated score 0.621


                     vibe  student  todays_route  diff_lo  diff_hi  claude
              light_heavy    0.894         0.890   -0.024    0.037   0.949
                   pacing    0.571         0.563   -0.109    0.134   0.710
        ending (accuracy)    0.636         0.667   -0.141    0.081   0.758
bittersweet found (of 11)    7.000         0.000      NaN      NaN   8.000
Claude: calls 21 of 99 bittersweet, 8 correct
student: calls 30 of 99 bittersweet, 7 correct
todays_route: calls 4 of 99 bittersweet, 0 correct


### Test C: the student reading TMDB text instead of Wikipedia

About 5,200 tier-3 movies have no Wikipedia text, only a TMDB overview and keywords. On the hidden movies that also have TMDB text, the same student reads the TMDB text instead, and its agreement with Claude is compared with its Wikipedia-text agreement on the same movies.

In [14]:
tmdb_dir, tmdb_of = ROOT / "data" / "raw" / "tmdb" / "movie_v1", tiers["tmdbId"]


def tmdb_text(m):
    f = tmdb_dir / f"{int(tmdb_of.get(m, -1)) if pd.notna(tmdb_of.get(m)) else -1}.json"
    d = json.loads(f.read_text()) if f.exists() else {}
    return ((d.get("overview") or "") + " " + " ".join(d.get("keywords") or [])).strip()


both = [m for m in hold if tmdb_text(m)]
via_tmdb = student_predict(student_hold_models, [tmdb_text(m) for m in both], both)
test_c = head_to_head(via_tmdb, student_hold.loc[both], teacher.loc[both], ("student_tmdb_text", "student_wiki_text"))
print(f"{len(both)} hidden movies with TMDB text")
print(test_c.round(3).to_string(index=False))

264 hidden movies with TMDB text
                     vibe  student_tmdb_text  student_wiki_text  diff_lo  diff_hi
              light_heavy              0.633              0.805   -0.244   -0.106
                   pacing              0.507              0.705   -0.286   -0.113
        ending (accuracy)              0.534              0.564   -0.087    0.027
bittersweet found (of 82)             34.000             38.000      NaN      NaN


### Part 3 result (2026-10-10)

**Claude labelled 2,000 movies** (the most-watched tier-3 movies with Wikipedia text, 74% of recent tier-3 ratings), with 0 broken answers, for $5.64. Total spend is $8.91.

| Test | Light↔Heavy | Pacing | Ending | Bittersweet |
|---|---|---|---|---|
| A. Agreement with Claude, 408 hidden movies: student vs today's route | 0.81 vs 0.81 (tie) | **0.71** vs 0.66 (better) | **58%** vs 51% (better) | **70** vs 24 of 130 found |
| B. Adil's 99 labels: student vs today's route | 0.89 vs 0.89 (tie) | 0.57 vs 0.56 (tie) | 64% vs 67% (tie) | 7 vs 0 of 11 found, but 30 vs 4 calls (23% vs 0% right) |
| B. Adil's 99 labels: Claude itself | **0.95** | **0.71** | **76%** | 8 of 11 found, 21 calls (38% right) |
| C. Student reading TMDB text vs Wikipedia text (82 movies) | 0.68 vs 0.79 | 0.48 vs 0.67 (clearly worse) | 48% vs 52% | |

**What it means**
- The student learned to imitate Claude better than today's route does (Test A), but on Adil's own labels it only ties today's route (Test B). Copying Claude through a word-counting model loses most of what made Claude better.
- It does find bittersweet endings, which today's route never does. But it calls bittersweet too often: 30 calls for 11 real ones.
- With only 99 labels, the intervals are about ±0.12 wide, so a small real gain could go unseen.
- Reading TMDB text is clearly weaker than reading Wikipedia, as in Step 5.
- **By the project's rule (a technique stays only if it beats the simpler baseline), the student has not earned its place on Adil's labels.** Claude's own answers have: for the 2,000 most-watched movies plus the 99 pilot movies, Claude's labels are the best available.

### Decision (Adil, 2026-10-10): option 1, and the saved table

One table holds Light↔Heavy (1–5, 5 = light), Pacing (1–5, 5 = fast) and Ending for every tier-3 movie, with a `source` column:
1. **adil:** his own labels for his 99 movies.
2. **claude:** Claude's answers for the 2,000 most-watched movies.
3. **todays_route:** everything else, from the movie's 41 vibe scores (`vibe_scores_v2`), using the Step 5 route fitted on Adil's 99 labels:
   - Light↔Heavy: ridge regression.
   - Ending: logistic regression.
   - Pacing: `fast − slow`, put on the 1–5 scale with a straight-line fit to Adil's pacing labels. This keeps the order of movies, so its Spearman score is unchanged.

Movies with no vibe scores (0.3%) stay empty. Claude and Adil give whole numbers; today's route gives in-between values such as 3.4.

In [15]:
THREE_PATH = P / "vibe_three_v1.parquet"
if THREE_PATH.exists():
    print(f"{THREE_PATH} already exists, loading.")
    vibe_three = pd.read_parquet(THREE_PATH)
else:
    from sklearn.linear_model import LinearRegression
    vibes = pd.read_parquet(P / "vibe_scores_v2.parquet").set_index("movieId")
    rest = [m for m in tier3 if m in vibes.index and m not in teacher.index and m not in human.index]
    f_lab, f_rest = vibes.loc[lid, ATTRS].astype(float), vibes.loc[rest, ATTRS].astype(float)
    lh = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25))).fit(f_lab, human["light_heavy"].astype(float))
    en = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=10, cv=5, max_iter=5000)).fit(f_lab, human["ending"])
    pace = LinearRegression().fit((f_lab["fast"] - f_lab["slow"]).to_frame(), human["pacing"].astype(float))
    route = pd.DataFrame({"light_heavy": np.clip(lh.predict(f_rest), 1, 5),
                          "pacing": np.clip(pace.predict((f_rest["fast"] - f_rest["slow"]).to_frame()), 1, 5),
                          "ending": en.predict(f_rest)}, index=rest).assign(source="todays_route")
    vibe_three = pd.concat([human.assign(source="adil"), teacher[["light_heavy", "pacing", "ending"]].assign(source="claude"), route])
    vibe_three = vibe_three.reindex(tier3)
    vibe_three["source"] = vibe_three["source"].fillna("none")
    vibe_three[["light_heavy", "pacing"]] = vibe_three[["light_heavy", "pacing"]].astype(float)
    vibe_three.index.name = "movieId"
    vibe_three = vibe_three.reset_index()
    vibe_three.to_parquet(THREE_PATH, index=False)
    (P / "vibe_three_v1.metadata.json").write_text(json.dumps({
        "created_at": pd.Timestamp.now(tz="UTC").isoformat(), "produced_by": "notebooks/13_bedrock.ipynb",
        "rows": len(vibe_three), "sources": vibe_three["source"].value_counts().to_dict(),
        "scales": {"light_heavy": "1-5, 5 = light", "pacing": "1-5, 5 = fast", "ending": ["happy", "bittersweet", "sad", "open"]},
        "inputs": ["data/processed/vibe_labels_v2.parquet", "data/raw/bedrock/vibe_three_v1/", "data/processed/vibe_scores_v2.parquet"],
        "route": "ridge (Light/Heavy) and logistic (Ending) on the 41 vibe scores, fitted on Adil's 99 labels; Pacing = linear(fast - slow)",
        "evidence": "Part 2/3 of this notebook; docs/DECISIONS.md 2026-10-10", "llm_cost_usd": 5.64}, indent=1))
    print(f"saved {len(vibe_three):,} rows to {THREE_PATH}")
print(vibe_three["source"].value_counts().to_dict())
print(vibe_three.groupby("source")["ending"].value_counts(normalize=True).unstack().round(2))

../data/processed/vibe_three_v1.parquet already exists, loading.
{'todays_route': 32203, 'claude': 2000, 'none': 101, 'adil': 99}
ending        bittersweet  happy  open   sad
source                                      
adil                 0.11   0.58  0.12  0.19
claude               0.35   0.46  0.05  0.14
todays_route         0.00   0.97  0.01  0.02


## Part 4: tier 2 in search (decided with Adil 2026-10-10)

**Why.** PROJECT.md §4.3 item 9 planned for the ranker to learn the tier-2 weight. That cannot work on MovieLens:
- Every tier-2 movie has 1–2 ratings in total.
- Tier 2 gets 0.05% (validation period) and 0.14% (test period) of all ratings, with each movie rated once at most.
- ALS has no signal for these movies, so it never proposes them as candidates.

So tier 2 stays out of the personalised feed. The plan is to make it searchable instead, where the match on vibe and text decides and the two-stage popularity step keeps it below well-known films. No weight is set by hand.

**Test.** Adil's 30 requests with the LLM-B parser (cached answers, so free), searching tier 3 only vs tiers 2 + 3:
- Does the expected film still appear?
- How many tier-2 films reach the top 10 and the top 50?

Tier 2 stays searchable only if results do not get worse.

In [16]:
engine23 = SearchEngine(ROOT, encoder, tiers=(2, 3))
tier_of = engine23.tiers["tier"]
print(f"searchable: tier 3 only {len(engine.ids):,} movies; tiers 2 + 3 {len(engine23.ids):,}")
rows = []
for r in reqs.itertuples():
    exp = any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(r.expected)))) if isinstance(r.expected, str) else None
    for name, eng in {"tier 3 only": engine, "tiers 2 + 3": engine23}.items():
        recs = eng.rank(eng.from_llm(llm(r.request)), k=50, vibe_mode="llm_attrs")
        rows.append({"n": r.n, "search": name, "expected_in_top10": exp in recs[:10] if exp else np.nan,
                     "expected_in_top50": exp in recs if exp else np.nan,
                     "tier2_in_top10": int((tier_of.reindex(recs[:10]) == 2).sum()),
                     "tier2_in_top50": int((tier_of.reindex(recs) == 2).sum()),
                     "same_top10": np.nan, "recs": recs})
tier2_eval = pd.DataFrame(rows)
a = tier2_eval[tier2_eval["search"] == "tier 3 only"].set_index("n")["recs"]
b = tier2_eval[tier2_eval["search"] == "tiers 2 + 3"].set_index("n")["recs"]
tier2_eval.loc[tier2_eval["search"] == "tiers 2 + 3", "same_top10"] = [len(set(a[n][:10]) & set(b[n][:10])) / 10 for n in b.index]
print(tier2_eval.groupby("search")[["expected_in_top10", "expected_in_top50", "tier2_in_top10", "tier2_in_top50", "same_top10"]]
      .mean().round(3).to_string())
for r in tier2_eval[(tier2_eval["search"] == "tiers 2 + 3") & (tier2_eval["tier2_in_top50"] > 0)].itertuples():
    t2 = [engine23.tiers["title"].get(m) for m in r.recs if tier_of.get(m) == 2]
    print(f"#{r.n}: tier-2 films in top 50 at ranks {[r.recs.index(m) + 1 for m in r.recs if tier_of.get(m) == 2]}: {'; '.join(t2)}")
tier2_eval.drop(columns="recs").to_csv(ARTIFACTS_DIR / "search_tier2_eval_v1.csv", index=False)

searchable: tier 3 only 34,302 movies; tiers 2 + 3 39,450


            expected_in_top10 expected_in_top50  tier2_in_top10  tier2_in_top50  same_top10
search                                                                                     
tier 3 only          0.464286          0.678571             0.0           0.000         NaN
tiers 2 + 3          0.464286          0.678571             0.0           1.767        0.97
#2: tier-2 films in top 50 at ranks [30, 35, 48]: Imagine You & Me (2016); Fractured Follies (1988); The Biggest Fan (2002)
#4: tier-2 films in top 50 at ranks [15, 19, 33, 43, 44, 45, 46, 50]: Fractured Follies (1988); Eamon (2010); Imagine You & Me (2016); Jack, Jules, Esther, & Me (2013); Duchess Of Idaho (1950); Queenie in Love (2001); Bengal Tiger (2015); Design for Scandal (1941)
#6: tier-2 films in top 50 at ranks [21, 44, 46, 47]: Pitbull: Last Dog (2018); Le Grand pardon (1982); The Violent Professionals (1973); Contraband (1980)
#7: tier-2 films in top 50 at ranks [25, 42]: Salt and Pepper (1968); One Night in the 